In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import f1_score, accuracy_score
import xgboost as xgb
import optuna
import lightgbm as lgb

In [ ]:
df = pd.read_csv('../data/002_processed_data/xy_df_hopv.csv')
print(df.shape)
print(df['stability'].shape)
print(df['stability'].isna().sum())
df = df.dropna(subset=['stability']) 
print(df['stability'].isna().sum())
X = df[[column for column in df if column not in ['smiles', 'stability', 'homo', 'lumo']]]
feature_columns = X.columns.tolist()
y = df['stability']
le = LabelEncoder()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Try different k values to see what works
for k in [20, 30, 40, 50]:
    selector = SelectKBest(f_classif, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        eval_metric='mlogloss',
        random_state=42
    )
    
    accuracy = cross_val_score(xgb_model, X_train_selected, y_train, cv=cv, scoring='accuracy').mean()
    f1 = cross_val_score(xgb_model, X_train_selected, y_train, cv=cv, scoring='f1_macro').mean()
    
    print(f'k={k}: Accuracy={accuracy:.3f}, F1={f1:.3f}')

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Remove constant features before anything else
var_selector = VarianceThreshold(threshold=0)
X_train_var = var_selector.fit_transform(X_train)
print(f"Features after removing constants: {X_train_var.shape[1]}")  # Should be ~172

In [ ]:
# Remove constants first
var_selector = VarianceThreshold(threshold=0)
X_train_clean = var_selector.fit_transform(X_train)

# Then select best
selector = SelectKBest(f_classif, k=40)
X_train_selected = selector.fit_transform(X_train_clean, y_train)

xgb_model = xgb.XGBClassifier(
    eval_metric='mlogloss',
    random_state=42
)

accuracy = cross_val_score(xgb_model, X_train_selected, y_train, cv=cv, scoring='accuracy').mean()
print(f'Accuracy: {accuracy:.3f}')

In [ ]:
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
data = pd.read_csv('../data/003_kaggle_competition_data/25261_classification_sample_submission.csv')
smiles = data['smiles']
smiles

In [ ]:
descriptor_names = [x[0] for x in Descriptors._descList]


In [ ]:
def compute_descriptors(smiles_str):
    """
    WRITE A DESCRIPTION

    Parameters
    ----------
    smiles_str : str
        The SMILES string representation of a molecule.

    Returns
    -------
    tuple
        A tuple containing the values of all available molecular descriptors
        (217 descriptors in total, from RDKit descriptor list).
    """
    # YOUR CODE HERE
    mol = Chem.MolFromSmiles(smiles_str)
    features = tuple(getattr(Descriptors, name)(mol) for name in descriptor_names)
    return features


In [ ]:
descriptors_df = pd.DataFrame(smiles.apply(compute_descriptors).tolist(), columns=descriptor_names)
descriptors_features = descriptors_df.columns.tolist()
descriptors_df

In [ ]:
# Fit preprocessors on full data
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

var_selector = VarianceThreshold(threshold=0)
X_clean = var_selector.fit_transform(X_imputed)

y = le.transform(y)
selector = SelectKBest(f_classif, k=40)
X_selected = selector.fit_transform(X_clean, y)

xgb_model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=456)
xgb_model.fit(X_selected, y)

X_test_imputed = imputer.transform(descriptors_df)
X_test_clean = var_selector.transform(X_test_imputed)   
X_test_selected = selector.transform(X_test_clean)


In [ ]:
y_pred = xgb_model.predict(X_test_selected)
y_pred = le.inverse_transform(y_pred)
submission_df = pd.DataFrame({
    'smiles': smiles,
    'stability': y_pred
})
pd.DataFrame(submission_df).to_csv('../data/003_kaggle_competition_data/25261_classification_sample_submission.csv', index=False)
